In [1]:
from __future__ import annotations

import os
from getpass import getpass

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery, QueryReference

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
for collection_name in ["ManualReview", "ManualProduct"]:
    if client.collections.exists(collection_name):
        client.collections.delete(collection_name)
        print("Deleted:", collection_name)

In [5]:
products = client.collections.create(
    name="ManualProduct",
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="name",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="category",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="description",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="price_pln",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="target_user",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created ManualProduct")

Created ManualProduct


In [6]:
reviews = client.collections.create(
    name="ManualReview",
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    properties=[
        wvc.config.Property(
            name="review_text",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="rating",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="reviewer_role",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="sentiment",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
    references=[
        wvc.config.ReferenceProperty(
            name="forProduct",
            target_collection="ManualProduct",
        ),
    ],
)

print("Created ManualReview")

Created ManualReview


In [7]:
product_data = [
    {
        "name": "AI Pro Workstation",
        "category": "AI Workstation",
        "description": (
            "Powerful workstation for local LLM experiments, CUDA workloads, "
            "computer vision, vector databases and data science projects."
        ),
        "price_pln": 7900.0,
        "target_user": "AI developer",
    },
    {
        "name": "Backend Developer PC",
        "category": "Developer Workstation",
        "description": (
            "Reliable development machine for FastAPI, Docker, PostgreSQL, testing, "
            "CI simulation and daily backend engineering."
        ),
        "price_pln": 6100.0,
        "target_user": "Backend developer",
    },
    {
        "name": "Office Mini PC",
        "category": "Office PC",
        "description": (
            "Compact office computer for email, documents, web applications, "
            "video meetings and light scripting tasks."
        ),
        "price_pln": 2600.0,
        "target_user": "Office worker",
    },
    {
        "name": "Deep Learning Tower",
        "category": "AI Workstation",
        "description": (
            "High-end tower for deep learning, large models, GPU acceleration, "
            "CUDA development and heavy AI workloads."
        ),
        "price_pln": 15500.0,
        "target_user": "Machine learning engineer",
    },
]

In [8]:
product_uuids = {}

for product in product_data:
    product_uuid = products.data.insert(product)
    product_uuids[product["name"]] = product_uuid

    print(product["name"], "->", product_uuid)

AI Pro Workstation -> 25801e31-67be-4896-98a4-543a9e054bf5
Backend Developer PC -> 16a97197-3b8f-4606-b570-1af6280ae3d9
Office Mini PC -> ff277fb8-1f41-450b-b67b-582de93252cd
Deep Learning Tower -> ad38aff3-7e61-4b84-b0f6-cbbaeed61f0c


In [9]:
review_data = [
    {
        "product_name": "AI Pro Workstation",
        "review_text": (
            "Excellent machine for local AI experiments. The GPU performance is strong, "
            "and 64 GB RAM is enough for many notebooks and vector database tests."
        ),
        "rating": 5,
        "reviewer_role": "AI engineer",
        "sentiment": "positive",
    },
    {
        "product_name": "AI Pro Workstation",
        "review_text": (
            "Good balance between price and performance for CUDA development. "
            "It is not as powerful as a 4090 tower, but much more affordable."
        ),
        "rating": 4,
        "reviewer_role": "data scientist",
        "sentiment": "positive",
    },
    {
        "product_name": "Backend Developer PC",
        "review_text": (
            "Very good for FastAPI, Docker, PostgreSQL and test automation. "
            "The lack of dedicated GPU is not a problem for backend work."
        ),
        "rating": 5,
        "reviewer_role": "backend developer",
        "sentiment": "positive",
    },
    {
        "product_name": "Backend Developer PC",
        "review_text": (
            "Great CPU and RAM for daily development, but not suitable for serious local AI workloads."
        ),
        "rating": 4,
        "reviewer_role": "software engineer",
        "sentiment": "mixed",
    },
    {
        "product_name": "Office Mini PC",
        "review_text": (
            "Cheap and quiet machine for office work, email and video meetings. "
            "Not designed for development or AI workloads."
        ),
        "rating": 4,
        "reviewer_role": "office manager",
        "sentiment": "positive",
    },
    {
        "product_name": "Deep Learning Tower",
        "review_text": (
            "Outstanding performance for large AI models and heavy CUDA workloads, "
            "but the price is high and it is excessive for beginners."
        ),
        "rating": 5,
        "reviewer_role": "ML engineer",
        "sentiment": "mixed",
    },
]

In [10]:
review_uuids = []

for review in review_data:
    product_name = review["product_name"]

    review_properties = {
        "review_text": review["review_text"],
        "rating": review["rating"],
        "reviewer_role": review["reviewer_role"],
        "sentiment": review["sentiment"],
    }

    review_uuid = reviews.data.insert(review_properties)
    review_uuids.append(review_uuid)

    reviews.data.reference_add(
        from_uuid=review_uuid,
        from_property="forProduct",
        to=product_uuids[product_name],
    )

    print("Review:", review_uuid)
    print("Linked product:", product_name)
    print("-" * 80)

Review: 199827be-fcb9-4fe0-a56e-874248f14424
Linked product: AI Pro Workstation
--------------------------------------------------------------------------------
Review: ae356d0c-2202-480f-8b7b-e777ff68768e
Linked product: AI Pro Workstation
--------------------------------------------------------------------------------
Review: f71a1fd5-8aac-48be-b177-ce4ef946ed59
Linked product: Backend Developer PC
--------------------------------------------------------------------------------
Review: 6bba1627-b0b2-4c53-af12-9911b2ca8980
Linked product: Backend Developer PC
--------------------------------------------------------------------------------
Review: 3bb409ad-7051-46dc-a413-6c84ce23b2c2
Linked product: Office Mini PC
--------------------------------------------------------------------------------
Review: 48f1f804-804d-4916-ac77-061e97625bcb
Linked product: Deep Learning Tower
--------------------------------------------------------------------------------


In [11]:
response = products.query.near_text(
    query="computer for local AI experiments and vector database work",
    limit=3,
    return_properties=[
        "name",
        "category",
        "description",
        "price_pln",
        "target_user",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("UUID:", obj.uuid)
    print("Name:", obj.properties["name"])
    print("Category:", obj.properties["category"])
    print("Price:", obj.properties["price_pln"])
    print("Target user:", obj.properties["target_user"])
    print("Description:", obj.properties["description"])
    print("-" * 80)

Distance: 0.3090256452560425
UUID: 25801e31-67be-4896-98a4-543a9e054bf5
Name: AI Pro Workstation
Category: AI Workstation
Price: 7900.0
Target user: AI developer
Description: Powerful workstation for local LLM experiments, CUDA workloads, computer vision, vector databases and data science projects.
--------------------------------------------------------------------------------
Distance: 0.4302842617034912
UUID: ad38aff3-7e61-4b84-b0f6-cbbaeed61f0c
Name: Deep Learning Tower
Category: AI Workstation
Price: 15500.0
Target user: Machine learning engineer
Description: High-end tower for deep learning, large models, GPU acceleration, CUDA development and heavy AI workloads.
--------------------------------------------------------------------------------
Distance: 0.5508183240890503
UUID: 16a97197-3b8f-4606-b570-1af6280ae3d9
Name: Backend Developer PC
Category: Developer Workstation
Price: 6100.0
Target user: Backend developer
Description: Reliable development machine for FastAPI, Docker, Po

In [12]:
best_product = response.objects[0]

best_product_uuid = best_product.uuid
best_product_name = best_product.properties["name"]

print("Best product:", best_product_name)
print("UUID:", best_product_uuid)

Best product: AI Pro Workstation
UUID: 25801e31-67be-4896-98a4-543a9e054bf5


In [13]:
response = reviews.query.fetch_objects(
    filters=Filter.by_ref("forProduct").by_id().equal(best_product_uuid),
    limit=10,
    return_properties=[
        "review_text",
        "rating",
        "reviewer_role",
        "sentiment",
    ],
    return_references=[
        QueryReference(
            link_on="forProduct",
            return_properties=[
                "name",
                "category",
                "price_pln",
            ],
        )
    ],
)

for obj in response.objects:
    print("Review UUID:", obj.uuid)
    print("Rating:", obj.properties["rating"])
    print("Reviewer role:", obj.properties["reviewer_role"])
    print("Sentiment:", obj.properties["sentiment"])
    print("Review:", obj.properties["review_text"])

    for product_ref in obj.references["forProduct"].objects:
        print("Product:", product_ref.properties["name"])
        print("Category:", product_ref.properties["category"])
        print("Price:", product_ref.properties["price_pln"])

    print("-" * 80)

Review UUID: 199827be-fcb9-4fe0-a56e-874248f14424
Rating: 5
Reviewer role: AI engineer
Sentiment: positive
Review: Excellent machine for local AI experiments. The GPU performance is strong, and 64 GB RAM is enough for many notebooks and vector database tests.
Product: AI Pro Workstation
Category: AI Workstation
Price: 7900.0
--------------------------------------------------------------------------------
Review UUID: ae356d0c-2202-480f-8b7b-e777ff68768e
Rating: 4
Reviewer role: data scientist
Sentiment: positive
Review: Good balance between price and performance for CUDA development. It is not as powerful as a 4090 tower, but much more affordable.
Product: AI Pro Workstation
Category: AI Workstation
Price: 7900.0
--------------------------------------------------------------------------------


In [14]:
def build_product_context(product_obj, review_objects) -> str:
    product = product_obj.properties

    lines = [
        "PRODUCT",
        f"Name: {product['name']}",
        f"Category: {product['category']}",
        f"Target user: {product['target_user']}",
        f"Price PLN: {product['price_pln']}",
        f"Description: {product['description']}",
        "",
        "REVIEWS",
    ]

    for number, review_obj in enumerate(review_objects, start=1):
        review = review_obj.properties

        lines.extend(
            [
                f"Review {number}:",
                f"Rating: {review['rating']}",
                f"Reviewer role: {review['reviewer_role']}",
                f"Sentiment: {review['sentiment']}",
                f"Text: {review['review_text']}",
                "",
            ]
        )

    return "\n".join(lines)

In [15]:
product_context = build_product_context(
    product_obj=best_product,
    review_objects=response.objects,
)

print(product_context)

PRODUCT
Name: AI Pro Workstation
Category: AI Workstation
Target user: AI developer
Price PLN: 7900.0
Description: Powerful workstation for local LLM experiments, CUDA workloads, computer vision, vector databases and data science projects.

REVIEWS
Review 1:
Rating: 5
Reviewer role: AI engineer
Sentiment: positive
Text: Excellent machine for local AI experiments. The GPU performance is strong, and 64 GB RAM is enough for many notebooks and vector database tests.

Review 2:
Rating: 4
Reviewer role: data scientist
Sentiment: positive
Text: Good balance between price and performance for CUDA development. It is not as powerful as a 4090 tower, but much more affordable.



In [16]:
CONTEXT_COLLECTION_NAME = "ManualContextAnswer"

if client.collections.exists(CONTEXT_COLLECTION_NAME):
    client.collections.delete(CONTEXT_COLLECTION_NAME)

context_answers = client.collections.create(
    name=CONTEXT_COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="question",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="context",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created ManualContextAnswer")

Created ManualContextAnswer


In [17]:
question = "Is this product a good choice for local AI experiments and vector database work?"

context_uuid = context_answers.data.insert(
    {
        "question": question,
        "context": product_context,
    }
)

print("Context object UUID:", context_uuid)

Context object UUID: 8924f766-a29f-402d-8b79-e7e904925064


In [18]:
response = context_answers.generate.near_text(
    query=question,
    single_prompt=(
        "Answer the question using only this context.\n\n"
        "Question: {question}\n\n"
        "Context:\n{context}\n\n"
        "Give a concise recommendation. Mention strengths, weaknesses and price."
    ),
    limit=1,
    return_properties=[
        "question",
        "context",
    ],
)

for obj in response.objects:
    print(obj.generated)

The AI Pro Workstation is a strong choice for local AI experiments and vector database work, priced at PLN 7900. Its notable strengths include powerful GPU performance, ample 64 GB RAM for various workloads, and a good balance between price and performance for CUDA development. However, it may not match the power of higher-end models like the 4090 tower. Overall, it is well-suited for AI developers looking for a robust yet affordable solution.


In [19]:
def answer_about_best_product(product_query: str, question: str) -> None:
    product_response = products.query.near_text(
        query=product_query,
        limit=1,
        return_properties=[
            "name",
            "category",
            "description",
            "price_pln",
            "target_user",
        ],
        return_metadata=MetadataQuery(distance=True),
    )

    if not product_response.objects:
        print("No product found.")
        return

    product_obj = product_response.objects[0]

    review_response = reviews.query.fetch_objects(
        filters=Filter.by_ref("forProduct").by_id().equal(product_obj.uuid),
        limit=10,
        return_properties=[
            "review_text",
            "rating",
            "reviewer_role",
            "sentiment",
        ],
    )

    context = build_product_context(
        product_obj=product_obj,
        review_objects=review_response.objects,
    )

    context_uuid = context_answers.data.insert(
        {
            "question": question,
            "context": context,
        }
    )

    answer_response = context_answers.generate.near_text(
        query=question,
        single_prompt=(
            "Answer the question using only this context.\n\n"
            "Question: {question}\n\n"
            "Context:\n{context}\n\n"
            "Give a practical recommendation. Mention product fit, limitations and price."
        ),
        limit=1,
        return_properties=[
            "question",
            "context",
        ],
    )

    print("PRODUCT QUERY:", product_query)
    print("QUESTION:", question)
    print("CONTEXT UUID:", context_uuid)
    print("-" * 80)

    for obj in answer_response.objects:
        print(obj.generated)

In [20]:
answer_about_best_product(
    product_query="workstation for local LLM experiments and vector database testing",
    question="Should I buy this product for learning Weaviate, local AI experiments and Python notebooks?",
)

PRODUCT QUERY: workstation for local LLM experiments and vector database testing
QUESTION: Should I buy this product for learning Weaviate, local AI experiments and Python notebooks?
CONTEXT UUID: a32870f9-079c-4329-bb0e-0423e07d763d
--------------------------------------------------------------------------------
The AI Pro Workstation is a suitable choice for local AI experiments and vector database work. Its powerful specifications, including a strong GPU performance and 64 GB of RAM, make it well-suited for tasks like local LLM experiments, CUDA workloads, and data science projects, as highlighted in the positive reviews from an AI engineer and a data scientist.

In terms of limitations, while the workstation offers good performance, it is noted that it may not reach the same power level as a 4090 tower. However, it strikes a commendable balance between price and performance for developers working on local AI initiatives.

At a price of PLN 7900.0, the workstation presents a cost-ef

In [21]:
answer_about_best_product(
    product_query="computer for FastAPI Docker PostgreSQL and backend development",
    question="Is this product a good choice for backend Python development?",
)

PRODUCT QUERY: computer for FastAPI Docker PostgreSQL and backend development
QUESTION: Is this product a good choice for backend Python development?
CONTEXT UUID: 9307081d-2ef4-40a9-b58d-ab06052c3a16
--------------------------------------------------------------------------------
Based on the provided context, the AI Pro Workstation is a suitable choice for learning Weaviate, conducting local AI experiments, and working with Python notebooks. 

**Product Fit:**
- The workstation is specifically designed for AI developers, which aligns with your learning and experimentation goals.
- It excels in local AI experiments and supports CUDA workloads, making it ideal for developing AI models and working with data science projects.
- With 64 GB of RAM, it can efficiently handle multiple Python notebooks and vector database tests.

**Limitations:**
- While the workstation has strong GPU performance, it may not match the capabilities of higher-end options like a 4090 tower. If you anticipate nee

In [22]:
answer_about_best_product(
    product_query="cheap office computer for email documents and meetings",
    question="Is this product suitable for office work and basic business tasks?",
)

PRODUCT QUERY: cheap office computer for email documents and meetings
QUESTION: Is this product suitable for office work and basic business tasks?
CONTEXT UUID: 0da71d78-a267-40d1-a586-9a0a8ad32fb3
--------------------------------------------------------------------------------
The AI Pro Workstation is a solid choice for local AI experiments and vector database work, especially for AI developers. Priced at PLN 7900, it offers a good balance between performance and affordability, particularly for CUDA workloads and projects involving computer vision and data science.

Fit: The workstation features strong GPU performance and 64 GB of RAM, which is ideal for many LLM experiments, vector database tests, and data-heavy applications. Positive reviews from both an AI engineer and a data scientist highlight its effectiveness in these areas.

Limitations: While it is not as powerful as high-end options like the 4090 tower, it provides substantial capabilities at a more accessible price point. 

In [23]:
response = products.generate.near_text(
    query="workstation for local AI and vector database work",
    grouped_task=(
        "Recommend the best product using only the retrieved product objects. "
        "Mention price and target user."
    ),
    limit=3,
    return_properties=[
        "name",
        "category",
        "description",
        "price_pln",
        "target_user",
    ],
)

print(response.generated)

print("\nSources:")
for obj in response.objects:
    print("-", obj.properties["name"], "|", obj.properties["price_pln"], "PLN")

The best product for AI development tasks is the **AI Pro Workstation**. 

- **Price**: 7900 PLN
- **Target User**: AI Developer

This workstation is designed specifically for local LLM experiments, CUDA workloads, computer vision, vector databases, and data science projects, making it an ideal choice for AI developers looking for robust performance at a competitive price.

Sources:
- AI Pro Workstation | 7900.0 PLN
- Deep Learning Tower | 15500.0 PLN
- Backend Developer PC | 6100.0 PLN


In [24]:
print(
    "Product-only generation uses only product fields. "
    "Manual context generation can combine product data with related review data, "
    "so the final answer has richer context."
)

Product-only generation uses only product fields. Manual context generation can combine product data with related review data, so the final answer has richer context.


In [25]:
client.close()